# Securing Text-to-SQL with Claude

Natural language to SQL is powerful, but executing AI-generated SQL against a real database introduces serious security risks. A naive implementation lets users (or prompt injections) read other tenants' data, drop tables, or exfiltrate secrets.

This notebook builds a [defense-in-depth](https://en.wikipedia.org/wiki/Defense_in_depth_(computing)) security layer for text-to-SQL, adding:

1. **Query validation** to allow only SELECT statements and block destructive operations
2. **Tenant scoping** with automatic [CTE](https://www.sqlite.org/lang_with.html)-based row filtering so users only see their own data
3. **Output sanitization** to strip sensitive columns (API keys, payment IDs) before returning results
4. **Operational guardrails** for LIMIT capping and query timeouts

We start with a naive implementation, demonstrate eight real attacks against it, then build each defense layer and show that the attacks are blocked.

> **Prerequisite:** This is a companion to the [Text-to-SQL guide](guide.ipynb), which covers prompt engineering for SQL generation. Read that first if you're new to text-to-SQL with Claude.

## Setup

In [1]:
%%capture
%pip install anthropic

In [ ]:
import sqlite3

from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-sonnet-4-6"

## Building a multi-tenant database

We'll use a fitness coaching platform as our example: a SaaS app where multiple trainers each manage their own clients, sessions, and invoices. This is a realistic [multi-tenant](https://en.wikipedia.org/wiki/Multitenancy) scenario where **data isolation between trainers is critical**.

The schema includes deliberately sensitive columns (`api_token`, `stripe_account_id`, `stripe_customer_id`, `stripe_payment_intent_id`) that should never be exposed through a natural language query interface.

In [3]:
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE trainers (
    id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    stripe_account_id TEXT,
    api_token TEXT
);

CREATE TABLE clients (
    id TEXT PRIMARY KEY,
    trainer_id TEXT NOT NULL REFERENCES trainers(id),
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    phone TEXT,
    stripe_customer_id TEXT
);

CREATE TABLE sessions (
    id TEXT PRIMARY KEY,
    client_id TEXT NOT NULL REFERENCES clients(id),
    trainer_id TEXT NOT NULL REFERENCES trainers(id),
    date TEXT NOT NULL,
    duration_minutes INTEGER NOT NULL,
    status TEXT NOT NULL CHECK (status IN ('scheduled', 'completed', 'cancelled')),
    notes TEXT,
    rate_pence INTEGER NOT NULL
);

CREATE TABLE invoices (
    id TEXT PRIMARY KEY,
    client_id TEXT NOT NULL REFERENCES clients(id),
    trainer_id TEXT NOT NULL REFERENCES trainers(id),
    amount_pence INTEGER NOT NULL,
    status TEXT NOT NULL CHECK (status IN ('draft', 'sent', 'paid', 'overdue')),
    stripe_payment_intent_id TEXT,
    created_at TEXT NOT NULL
);
""")

print("✓ Schema created: trainers, clients, sessions, invoices")

✓ Schema created: trainers, clients, sessions, invoices


### Seed data

Two trainers (Alice and Bob), five clients split between them, thirteen sessions, and five invoices. This gives us enough data to demonstrate cross-tenant leaks adequately.

In [4]:
# --- Trainers ---
cursor.executemany(
    "INSERT INTO trainers VALUES (?, ?, ?, ?, ?)",
    [
        (
            "t-alice",
            "Alice Johnson",
            "alice@fitpro.io",
            "acct_1A2B3C4D",
            "sk-alice-secret-token-999",
        ),
        ("t-bob", "Bob Martinez", "bob@fitpro.io", "acct_5E6F7G8H", "sk-bob-secret-token-888"),
    ],
)

# --- Clients (3 for Alice, 2 for Bob) ---
cursor.executemany(
    "INSERT INTO clients VALUES (?, ?, ?, ?, ?, ?)",
    [
        ("c-emma", "t-alice", "Emma Wilson", "emma@mail.com", "07700-100001", "cus_emma_001"),
        ("c-james", "t-alice", "James Chen", "james@mail.com", "07700-100002", "cus_james_002"),
        ("c-sofia", "t-alice", "Sofia Rossi", "sofia@mail.com", "07700-100003", "cus_sofia_003"),
        ("c-liam", "t-bob", "Liam O'Brien", "liam@mail.com", "07700-200001", "cus_liam_004"),
        ("c-nora", "t-bob", "Nora Ahmed", "nora@mail.com", "07700-200002", "cus_nora_005"),
    ],
)

# --- Sessions (8 for Alice's clients, 5 for Bob's clients) ---
cursor.executemany(
    "INSERT INTO sessions VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
    [
        # Alice's sessions
        (
            "s-01",
            "c-emma",
            "t-alice",
            "2025-01-06",
            60,
            "completed",
            "Deadlift form improved",
            5000,
        ),
        ("s-02", "c-emma", "t-alice", "2025-01-13", 60, "completed", "New squat PR", 5000),
        ("s-03", "c-james", "t-alice", "2025-01-07", 45, "completed", "Cardio baseline test", 4000),
        ("s-04", "c-james", "t-alice", "2025-01-14", 45, "cancelled", None, 4000),
        (
            "s-05",
            "c-sofia",
            "t-alice",
            "2025-01-08",
            30,
            "completed",
            "Flexibility assessment",
            3000,
        ),
        ("s-06", "c-sofia", "t-alice", "2025-01-15", 30, "completed", "Yoga flow intro", 3000),
        ("s-07", "c-emma", "t-alice", "2025-01-20", 60, "scheduled", None, 5000),
        ("s-08", "c-james", "t-alice", "2025-01-21", 45, "scheduled", None, 4000),
        # Bob's sessions
        ("s-09", "c-liam", "t-bob", "2025-01-06", 60, "completed", "Boxing drills", 5500),
        ("s-10", "c-liam", "t-bob", "2025-01-13", 60, "completed", "Sparring session", 5500),
        ("s-11", "c-nora", "t-bob", "2025-01-07", 45, "completed", "HIIT circuit", 4500),
        ("s-12", "c-nora", "t-bob", "2025-01-14", 45, "completed", "Endurance run", 4500),
        ("s-13", "c-liam", "t-bob", "2025-01-20", 60, "scheduled", None, 5500),
    ],
)

# --- Invoices ---
cursor.executemany(
    "INSERT INTO invoices VALUES (?, ?, ?, ?, ?, ?, ?)",
    [
        ("inv-01", "c-emma", "t-alice", 10000, "paid", "pi_emma_001", "2025-01-15"),
        ("inv-02", "c-james", "t-alice", 4000, "sent", "pi_james_002", "2025-01-15"),
        ("inv-03", "c-sofia", "t-alice", 6000, "draft", None, "2025-01-16"),
        ("inv-04", "c-liam", "t-bob", 11000, "paid", "pi_liam_003", "2025-01-15"),
        ("inv-05", "c-nora", "t-bob", 9000, "overdue", "pi_nora_004", "2025-01-15"),
    ],
)

conn.commit()
print(
    f"✓ Seeded: {cursor.execute('SELECT COUNT(*) FROM trainers').fetchone()[0]} trainers, "
    f"{cursor.execute('SELECT COUNT(*) FROM clients').fetchone()[0]} clients, "
    f"{cursor.execute('SELECT COUNT(*) FROM sessions').fetchone()[0]} sessions, "
    f"{cursor.execute('SELECT COUNT(*) FROM invoices').fetchone()[0]} invoices"
)

✓ Seeded: 2 trainers, 5 clients, 13 sessions, 5 invoices


### Auto-generate schema description

Rather than manually writing a schema description for the prompt, we extract it directly from the database. This ensures the description always matches the actual schema.

In [ ]:
def generate_schema_description(connection: sqlite3.Connection) -> str:
    """Extract a human-readable schema description from the database."""
    tables = connection.execute(
        "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()

    parts = []
    for table_name, _create_sql in tables:
        columns = connection.execute(f"PRAGMA table_info({table_name})").fetchall()
        row_count = connection.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

        col_descriptions = []
        for col in columns:
            _, col_name, col_type, not_null, default, pk = col
            desc = f"  - {col_name} ({col_type})"
            if pk:
                desc += " PRIMARY KEY"
            if not_null and not pk:
                desc += " NOT NULL"
            col_descriptions.append(desc)

        parts.append(f"Table: {table_name} ({row_count} rows)\n" + "\n".join(col_descriptions))

    return "\n\n".join(parts)


SCHEMA_DESCRIPTION = generate_schema_description(conn)
print(SCHEMA_DESCRIPTION)